# Velas japonesas y redes convolucionales
Trabajo final de Ciencia de Datos (A422), Universidad de San Andrés.

Este notebook no tiene código propio. Todo el código vive en la carpeta `src/` del repositorio y acá solo lo corremos, para aprovechar la GPU de Colab cuando entrenemos la red.

## Traer el repositorio
Clonamos el repo desde GitHub. Si ya estaba clonado, bajamos la última versión.

In [ ]:
%cd /content
!git clone -q https://github.com/ayerzafelix/tp-velas-cnn.git 2>/dev/null || git -C tp-velas-cnn pull -q
%cd /content/tp-velas-cnn
!pip install -q -r requirements.txt

## Descarga de precios
Bajamos velas semanales de 2005 a 2025 de 50 acciones del S&P 500, 5 por cada uno de 10 sectores, y de SPY como referencia. Los precios están ajustados por dividendos.

In [ ]:
!python src/descarga.py

## Calidad de los datos
Revisamos que el máximo de cada vela no quede por debajo de la apertura o el cierre, que el mínimo no quede por encima, que no haya precios en cero y si hay saltos semanales de más de 40%.

In [ ]:
!python src/calidad.py

## Estacionariedad
Una serie es estacionaria si su promedio y su variabilidad no cambian con el tiempo. Los precios no lo son, porque tienen tendencia. Los rendimientos deberían serlo.

Lo chequeamos con el test ADF sobre el logaritmo del precio y sobre el log-rendimiento semanal de cada serie. Si el p-valor da menos de 0,05 tomamos la serie como estacionaria.

In [ ]:
!python src/estacionariedad.py
from IPython.display import Image
Image("resultados/estacionariedad_spy.png")

## Etiquetas y balance de clases
Tomamos una muestra por acción cada 20 semanas, así las imágenes de una misma acción no comparten velas. Para cada muestra calculamos el log-rendimiento de la acción del cierre de la semana t al cierre de la semana t+8 y le restamos el de SPY en la misma ventana. Si da más de 0 la etiqueta es 1, o sea que la acción le ganó al S&P 500.

Después separamos por fecha en entrenamiento, validación y test, dejando al menos 28 semanas de hueco entre cada parte, y miramos qué porcentaje de cada grupo le gana al índice.

In [ ]:
!python src/etiquetas.py

## Imágenes de velas
Dibujamos una imagen por muestra con las 20 velas semanales que terminan en la semana t. Usamos el logaritmo de los precios y la escala va del mínimo al máximo de esas 20 velas, así el nivel de precio de la acción no influye. Las imágenes son de 64 x 64 píxeles, sin ejes, etiquetas ni volumen.

Las guardamos en `data/imagenes/`, separadas por conjunto y por clase, y mostramos algunas para revisarlas a ojo.

In [ ]:
!python src/imagenes.py
Image("resultados/ejemplos_imagenes.png")

## Baselines
Antes de entrenar la red armamos dos modelos simples para tener contra qué compararla. Si la red no les gana, no aporta.

El primero dice siempre la clase más frecuente del entrenamiento. El segundo es una regresión logística con dos números calculados sobre las mismas 20 semanas que ve la imagen: cuánto rindió la acción contra SPY y su volatilidad.

Los medimos en test con el AUC y con la precisión en la clase "le gana", o sea, de las veces que el modelo dice que la acción le gana al SPY, en cuántas acierta.

In [ ]:
!python src/baselines.py

## Primera red convolucional
Una red chica, parecida a las que armamos en clase: tres capas convolucionales con max pooling, una capa densa con dropout y una salida sigmoide que da la probabilidad de que la acción le gane al SPY.

Entrenamos con el conjunto de entrenamiento y usamos el de validación para frenar cuando deja de mejorar. El test lo usamos recién al final, con las mismas métricas que los baselines.

Conviene activar la GPU en Entorno de ejecución, Cambiar tipo de entorno de ejecución, T4 GPU.

In [ ]:
!python src/red.py
Image("resultados/curvas_red.png")

## Backtest
Simulamos invertir en el período de test siguiendo a cada modelo. En cada fecha compramos en partes iguales las acciones que el modelo marca como "le gana", las mantenemos 8 semanas y las vendemos, pagando 0,1% en cada compra y cada venta. Lo comparamos con comprar SPY en las mismas fechas y por el mismo tiempo.

Para la red mostramos además el resultado período por período y las compras que más aportaron, para ver si el rendimiento viene de muchas acciones o de unas pocas.

In [ ]:
!python src/backtest.py

## La red en 10 corridas
Al entrenar la red dos veces con el mismo código el backtest pasó de 54,7% a 5,6%. Con un resultado tan variable no se puede saber si un cambio mejora algo o si fue suerte.

Por eso entrenamos la misma red 10 veces con semillas distintas y miramos el promedio, el mínimo y el máximo de cada métrica. Desde ahora cualquier cambio a la red se compara así. Tarda unos minutos.

In [ ]:
!python src/varias_corridas.py

## Qué hace la red de verdad
La red no clasifica mejor que el azar pero gana plata en el backtest. Para entender por qué, y siguiendo lo que nos marcó un tutor, miramos tres cosas sobre las 10 corridas:

1. Tamaño del error. Predecir solo "le gana o no" esconde cuánto se equivoca la red. Para cada tipo de acierto y de error vemos cuánto rindieron en promedio esas acciones contra SPY.
2. Qué acciones elige. Comparamos la volatilidad de las que elige con la de las que descarta, y de qué sectores son sus compras.
3. Sesgo de selección. Vemos si comprar nuestras 50 acciones por igual, sin ninguna red, ya le gana al SPY. Si pasa, la comparación justa para la red es contra esas 50 y no solo contra SPY.

In [ ]:
!python src/analisis_red.py